In [1]:

%pip install pandas
%pip install ollama



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
#borrowing this code from Leon 

import os
import glob
import json
import gzip
import time 

import pandas as pd
from ollama import chat
from pydantic import BaseModel

In [2]:
## import transcript files and split each transcript into segment rows
from pathlib import Path
import re


def resolve_transcript_folder():
    candidates = [
        Path("transcripts"),
        Path("assignment 1/transcripts"),
        Path("../assignment 1/transcripts"),
    ]
    for folder in candidates:
        if folder.exists():
            return str(folder)
    return str(candidates[0])


def hms_to_seconds(hms: str) -> int:
    h, m, s = [int(x) for x in hms.split(":")]
    return h * 3600 + m * 60 + s


SEGMENT_RE = re.compile(r"^\[(\d{2}:\d{2}:\d{2})\s*-\s*(\d{2}:\d{2}:\d{2})\]\s*(.*)$")


def parse_transcript_segments(body: str):
    """Parse timestamped transcript text into segment dicts.
    Falls back to a single segment if no timestamps are found."""
    if not body or not body.strip():
        return []

    segments = []
    current = None

    for raw_line in body.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        match = SEGMENT_RE.match(line)
        if match:
            start_hms, end_hms, text = match.groups()
            current = {
                "start_hms": start_hms,
                "end_hms": end_hms,
                "start_sec": hms_to_seconds(start_hms),
                "end_sec": hms_to_seconds(end_hms),
                "parts": [text.strip()] if text.strip() else [],
            }
            segments.append(current)
        elif current is not None:
            current["parts"].append(line)

    if not segments:
        return [
            {
                "segment_id": "seg_0001",
                "start_hms": None,
                "end_hms": None,
                "start_sec": None,
                "end_sec": None,
                "segment_text": body.strip(),
            }
        ]

    clean_segments = []
    for idx, seg in enumerate(segments, start=1):
        segment_text = " ".join(p for p in seg["parts"] if p).strip()
        if not segment_text:
            continue
        clean_segments.append(
            {
                "segment_id": f"seg_{idx:04d}",
                "start_hms": seg["start_hms"],
                "end_hms": seg["end_hms"],
                "start_sec": seg["start_sec"],
                "end_sec": seg["end_sec"],
                "segment_text": segment_text,
            }
        )

    return clean_segments


def load_transcript(path):
    with open(path, encoding="utf-8") as f:
        raw = f.read()

    # header and body are separated by the first blank line
    header, _, body = raw.partition("\n\n")

    meta = {}
    for line in header.splitlines():
        key, _, value = line.partition(": ")
        meta[key.strip().lower()] = value.strip()

    segments = parse_transcript_segments(body.strip())

    return {
        "title": meta.get("title", ""),
        "date": meta.get("date", ""),
        "subcommittee": meta.get("subcommittee", ""),
        "url": meta.get("url", ""),
        "video_id": os.path.splitext(os.path.basename(path))[0].split("_")[-1],
        "file": path,
        "text": body.strip(),
        "segments": segments,
    }


def load_all(folder):
    paths = sorted(glob.glob(os.path.join(folder, "*.txt")))
    if not paths:
        print(f"No transcript .txt files found in: {folder}")
        return pd.DataFrame(columns=[
            "title", "date", "subcommittee", "url", "video_id", "file", "text", "segments", "word_count"
        ])

    df_local = pd.DataFrame([load_transcript(p) for p in paths])
    df_local["date"] = pd.to_datetime(df_local["date"], errors="coerce")
    df_local["word_count"] = df_local["text"].str.split().str.len()
    return df_local.sort_values("date").reset_index(drop=True)


def build_segment_rows(df_transcripts: pd.DataFrame) -> pd.DataFrame:
    segment_rows = []

    for _, row in df_transcripts.iterrows():
        for seg in row.get("segments", []):
            segment_rows.append(
                {
                    "video_id": row["video_id"],
                    "date": row["date"],
                    "title": row["title"],
                    "subcommittee": row["subcommittee"],
                    "url": row["url"],
                    "file": row["file"],
                    "segment_id": seg.get("segment_id"),
                    "start_hms": seg.get("start_hms"),
                    "end_hms": seg.get("end_hms"),
                    "start_sec": seg.get("start_sec"),
                    "end_sec": seg.get("end_sec"),
                    "segment_text": seg.get("segment_text", ""),
                }
            )

    df_seg = pd.DataFrame(segment_rows)
    if df_seg.empty:
        return df_seg

    df_seg["segment_text"] = df_seg["segment_text"].fillna("")
    df_seg["segment_word_count"] = df_seg["segment_text"].str.split().str.len()
    df_seg["row_id"] = df_seg["video_id"] + "__" + df_seg["segment_id"]
    return df_seg


ed_transcripts = resolve_transcript_folder()
print(f"Using transcripts folder: {ed_transcripts}")

df = load_all(ed_transcripts)
df_segments = build_segment_rows(df)

print("\nTranscript-level check:")
if df.empty:
    print("0 transcripts loaded")
else:
    print(df[["date", "video_id", "title", "word_count"]])
    print(f"{len(df)} transcripts loaded")

print("\nSegment-level check:")
if df_segments.empty:
    print("0 segment rows created")
else:
    print(df_segments[["video_id", "segment_id", "start_hms", "end_hms", "segment_word_count"]].head(12))
    print(f"{len(df_segments)} total segment rows")

Using transcripts folder: transcripts

Transcript-level check:
        date     video_id                                              title  \
0 2026-01-14  RM0aq5ynUiQ                     "Building an AI-Ready America"   
1 2026-02-11  Wd9cdenGMZI  "Building an AI-Ready America: Safer Workplace...   
2 2026-02-24  ZeJBjQY3LdI  "Building an AI-Ready America: Teaching in the...   
3 2026-03-04  8jANpuMrFG8  "Building an AI-Ready America: Strengthening E...   
4 2026-04-15  7QljNpZdu3Q  "Building an AI-Ready America: Understanding A...   
5 2026-06-03  kfRYHhh8yD4  "Building an AI-Ready America: Higher Educatio...   

   word_count  
0       29998  
1       10739  
2       16903  
3       16137  
4       10841  
5       17235  
6 transcripts loaded

Segment-level check:
       video_id segment_id start_hms   end_hms  segment_word_count
0   RM0aq5ynUiQ   seg_0001  00:04:00  00:04:30                  29
1   RM0aq5ynUiQ   seg_0002  00:04:30  00:05:00                  50
2   RM0aq5ynUiQ   se

In [3]:
## run keyword matching on each segment row (not full transcript)
import re

keywords_ai = [
    "artificial intelligence",
    "AI",
    "AI-assisted",
    "large language model",
    "large language models",
    "GenAI",
    "generative AI",
    "generative artificial intelligence",
    "LLM",
    "LLMs",
]

# escape + word boundaries to avoid weird partial matches
keyword_pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in keywords_ai) + r")\b",
    flags=re.IGNORECASE,
)

if "df_segments" not in globals() or df_segments.empty:
    raise ValueError("Expected non-empty df_segments. Re-run Cell 3 first.")

df_segments["segment_text"] = df_segments["segment_text"].fillna("")
df_segments["keyword_hits"] = df_segments["segment_text"].apply(lambda t: keyword_pattern.findall(t))
df_segments["keyword_hit_count"] = df_segments["keyword_hits"].str.len()
df_segments["contains_keywords_ai"] = df_segments["keyword_hit_count"] > 0

# transcript-level rollup for downstream comparisons
df_keyword_summary = (
    df_segments.groupby("video_id", as_index=False)
    .agg(
        keyword_hit_count=("keyword_hit_count", "sum"),
        segments_with_keywords=("contains_keywords_ai", "sum"),
        contains_keywords_ai=("contains_keywords_ai", "any"),
    )
)

# merge summary back to transcript-level df
for col in ["keyword_hit_count", "segments_with_keywords", "contains_keywords_ai"]:
    if col in df.columns:
        df = df.drop(columns=[col])
df = df.merge(df_keyword_summary, on="video_id", how="left")
df["keyword_hit_count"] = df["keyword_hit_count"].fillna(0).astype(int)
df["segments_with_keywords"] = df["segments_with_keywords"].fillna(0).astype(int)
df["contains_keywords_ai"] = df["contains_keywords_ai"].fillna(False).astype(bool)

print(f"Scanned {len(df_segments)} segments across {len(df)} transcripts")
print(f"Total keyword mentions found: {int(df_segments['keyword_hit_count'].sum())}")
print(f"Segments with >=1 mention: {int(df_segments['contains_keywords_ai'].sum())}")
print(f"Transcripts with >=1 segment mention: {int(df['contains_keywords_ai'].sum())}")

print("\nTop keyword-hit segments:")
display(
    df_segments.loc[df_segments["contains_keywords_ai"], [
        "video_id", "segment_id", "start_hms", "end_hms", "keyword_hit_count", "segment_text"
    ]]
    .sort_values("keyword_hit_count", ascending=False)
    .head(10)
)

print("\nTranscript-level keyword summary:")
display(
    df[["date", "video_id", "title", "keyword_hit_count", "segments_with_keywords", "contains_keywords_ai"]]
    .sort_values(["keyword_hit_count", "segments_with_keywords"], ascending=False)
    .head(10)
)

Scanned 1171 segments across 6 transcripts
Total keyword mentions found: 1315
Segments with >=1 mention: 751
Transcripts with >=1 segment mention: 6

Top keyword-hit segments:


,video_id,segment_id,start_hms,end_hms,keyword_hit_count,segment_text
157,RM0aq5ynUiQ,seg_0158,01:22:30,01:23:00,5,set of courses um that we're calling AI Founda...
890,7QljNpZdu3Q,seg_0051,00:25:00,00:25:30,5,as a foundation for an AI workforce research h...
113,RM0aq5ynUiQ,seg_0114,01:00:30,01:01:00,5,speak on what you believe will be the durable ...
286,RM0aq5ynUiQ,seg_0287,02:27:00,02:27:30,5,"student privacy? Yeah, first and foremost, ver..."
872,7QljNpZdu3Q,seg_0033,00:16:00,00:16:30,4,errors that have historically proved costly. A...
956,7QljNpZdu3Q,seg_0117,00:58:00,00:58:30,4,"Thank you, uh Ms. Beerbaugh. >> [snorts] >> As..."
1092,kfRYHhh8yD4,seg_0120,01:07:00,01:07:30,4,School of Law that adopted an AI policy prohib...
1086,kfRYHhh8yD4,seg_0114,01:04:00,01:04:30,4,of AI literacy in higher education and how are...
43,RM0aq5ynUiQ,seg_0044,00:25:30,00:26:00,4,writing feedback tool with them. She gets pers...
547,ZeJBjQY3LdI,seg_0080,01:04:00,01:04:30,4,your teachers arrive at their schools uh well ...



Transcript-level keyword summary:


,date,video_id,title,keyword_hit_count,segments_with_keywords,contains_keywords_ai
0,2026-01-14,RM0aq5ynUiQ,"""Building an AI-Ready America""",436,246,True
5,2026-06-03,kfRYHhh8yD4,"""Building an AI-Ready America: Higher Educatio...",247,139,True
2,2026-02-24,ZeJBjQY3LdI,"""Building an AI-Ready America: Teaching in the...",233,130,True
4,2026-04-15,7QljNpZdu3Q,"""Building an AI-Ready America: Understanding A...",181,98,True
1,2026-02-11,Wd9cdenGMZI,"""Building an AI-Ready America: Safer Workplace...",118,72,True
3,2026-03-04,8jANpuMrFG8,"""Building an AI-Ready America: Strengthening E...",100,66,True


In [ ]:
## Run each segment through Ollama, then roll up to transcript-level JSON per video
MODEL_NAME = "qwen3.5:4b"  # primary model
FALLBACK_MODEL = "qwen2.5:1.5b"  # optional smaller fallback (pull if needed)
MAX_CHARS_PER_SEGMENT = 2400
MAX_QUOTES_PER_SEGMENT = 3
MAX_QUOTES_PER_TRANSCRIPT = 12
OVERWRITE_EXISTING = True  # set False to keep existing JSON files
LLM_ONLY_KEYWORD_SEGMENTS = False  # True = cheaper/faster, False = full segment coverage

class AILabel(BaseModel):
    """Structured response from the model."""
    contains_ai_llm: bool
    quotes: list[str]


dir_labels = "output/ai_labels"
os.makedirs(dir_labels, exist_ok=True)


def run_labeling(prompt: str):
    """Try GPU first, then CPU mode, then smaller model."""
    attempts = [
        (MODEL_NAME, {"num_ctx": 2048}, "primary-gpu"),
        (MODEL_NAME, {"num_ctx": 1024, "num_gpu": 0}, "primary-cpu"),
        (FALLBACK_MODEL, {"num_ctx": 2048}, "fallback-gpu"),
        (FALLBACK_MODEL, {"num_ctx": 1024, "num_gpu": 0}, "fallback-cpu"),
    ]

    errors = []
    for model_name, options, mode in attempts:
        try:
            response = chat(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                format=AILabel.model_json_schema(),
                options=options,
            )
            return AILabel.model_validate_json(response.message.content), model_name, mode
        except Exception as e:
            errors.append(f"{mode} ({model_name}): {e}")

    raise RuntimeError(" | ".join(errors))


if "df_segments" not in globals() or df_segments.empty:
    raise ValueError("Expected non-empty df_segments. Re-run Cell 3 first.")

if LLM_ONLY_KEYWORD_SEGMENTS:
    if "contains_keywords_ai" not in df_segments.columns:
        raise ValueError("Run keyword cell first before enabling LLM_ONLY_KEYWORD_SEGMENTS.")
    df_llm = df_segments[df_segments["contains_keywords_ai"]].copy()
else:
    df_llm = df_segments.copy()

print(f"Labeling {len(df_llm)} segment rows with LLM...")

for video_id, video_segments in df_llm.groupby("video_id", sort=False):
    fn_out = os.path.join(dir_labels, f"{video_id}.json")

    if os.path.exists(fn_out) and not OVERWRITE_EXISTING:
        continue

    video_segments = video_segments.sort_values(["start_sec", "segment_id"], na_position="last")
    first_row = video_segments.iloc[0]

    segment_records = []
    failed_segments = []
    model_modes = set()
    models_used = set()

    print(f"\nVideo {video_id}: {len(video_segments)} segment(s)")

    for _, seg in video_segments.iterrows():
        segment_text = (seg.get("segment_text") or "").strip()
        if not segment_text:
            continue

        segment_text = segment_text[:MAX_CHARS_PER_SEGMENT]
        prompt = f"""Read the transcript segment below and decide if it clearly mentions AI or LLMs.
Return valid JSON with exactly these keys:
- contains_ai_llm (boolean)
- quotes (array of up to {MAX_QUOTES_PER_SEGMENT} short direct quotes from this segment; [] if no mention)

Rules:
- quotes must be verbatim text snippets from this segment only.
- do not include explanations outside JSON.

Segment metadata:
- video_id: {seg['video_id']}
- segment_id: {seg['segment_id']}
- start_hms: {seg.get('start_hms')}
- end_hms: {seg.get('end_hms')}

Transcript segment:
{segment_text}
"""

        try:
            label, used_model, run_mode = run_labeling(prompt)

            clean_quotes = []
            for q in (label.quotes or []):
                q2 = (q or "").strip()
                if q2 and q2 not in clean_quotes:
                    clean_quotes.append(q2)
            clean_quotes = clean_quotes[:MAX_QUOTES_PER_SEGMENT]

            segment_records.append(
                {
                    "segment_id": seg["segment_id"],
                    "start_hms": seg.get("start_hms"),
                    "end_hms": seg.get("end_hms"),
                    "start_sec": seg.get("start_sec"),
                    "end_sec": seg.get("end_sec"),
                    "contains_ai_llm": bool(label.contains_ai_llm),
                    "quote_count": len(clean_quotes),
                    "quotes": clean_quotes,
                    "segment_text_preview": segment_text[:220],
                    "model": used_model,
                    "run_mode": run_mode,
                }
            )
            model_modes.add(run_mode)
            models_used.add(used_model)
            print(f"  {seg['segment_id']}: ai={label.contains_ai_llm} quotes={len(clean_quotes)} via {used_model} ({run_mode})")

        except Exception as e:
            failed_segments.append({"segment_id": seg["segment_id"], "error": str(e)})
            print(f"  Failed segment {seg['segment_id']}: {e}")

    transcript_quotes = []
    for record in segment_records:
        for q in record.get("quotes", []):
            if q not in transcript_quotes:
                transcript_quotes.append(q)
    transcript_quotes = transcript_quotes[:MAX_QUOTES_PER_TRANSCRIPT]

    record = {
        "video_id": video_id,
        "date": str(first_row["date"].date()) if pd.notna(first_row["date"]) else None,
        "title": first_row["title"],
        "contains_ai_llm": any(r["contains_ai_llm"] for r in segment_records),
        "segment_count": len(segment_records),
        "segments_with_ai": int(sum(r["contains_ai_llm"] for r in segment_records)),
        "quote_count": len(transcript_quotes),
        "quotes": transcript_quotes,
        "quote": transcript_quotes[0] if transcript_quotes else "",  # backward compatibility
        "models_used": sorted(models_used),
        "run_modes_used": sorted(model_modes),
        "failed_segments": failed_segments,
        "segment_labels": segment_records,
    }

    with open(fn_out, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    print(f"Saved {video_id}: {record['segments_with_ai']}/{record['segment_count']} segments flagged AI")

Saved RM0aq5ynUiQ with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved Wd9cdenGMZI with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved ZeJBjQY3LdI with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved 8jANpuMrFG8 with qwen3.5:4b (primary-gpu) - 4 quote(s)
Saved 7QljNpZdu3Q with qwen3.5:4b (primary-gpu) - 5 quote(s)
Saved kfRYHhh8yD4 with qwen3.5:4b (primary-gpu) - 5 quote(s)


In [ ]:
labels = []
for fn in glob.glob(os.path.join(dir_labels, "*.json")):
    with open(fn, encoding="utf-8") as f:
        labels.append(json.load(f))

if not labels:
    print("No label files found yet. Run the Ollama labeling cell first.")
else:
    df_labels = pd.DataFrame(labels)

    # backward compatibility across old/new label formats
    if "quotes" not in df_labels.columns:
        df_labels["quotes"] = [[] for _ in range(len(df_labels))]
    if "quote" not in df_labels.columns:
        df_labels["quote"] = ""

    df_labels["quote_count"] = df_labels["quotes"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    print("Transcript-level LLM labels:")
    print(df_labels["contains_ai_llm"].value_counts(dropna=False))
    print("\nQuote count summary:")
    print(df_labels["quote_count"].describe())

    # flatten segment labels for segment-level QA and analysis
    segment_rows = []
    for rec in labels:
        for seg in rec.get("segment_labels", []):
            segment_rows.append(
                {
                    "video_id": rec.get("video_id"),
                    "title": rec.get("title"),
                    "segment_id": seg.get("segment_id"),
                    "start_hms": seg.get("start_hms"),
                    "end_hms": seg.get("end_hms"),
                    "contains_ai_llm": seg.get("contains_ai_llm"),
                    "quote_count": seg.get("quote_count", 0),
                    "quotes": seg.get("quotes", []),
                    "segment_text_preview": seg.get("segment_text_preview", ""),
                    "model": seg.get("model"),
                    "run_mode": seg.get("run_mode"),
                }
            )

    df_segment_labels = pd.DataFrame(segment_rows)
    if df_segment_labels.empty:
        print("\nNo segment_labels found in JSON files (older format likely).")
    else:
        print("\nSegment-level LLM labels:")
        print(df_segment_labels["contains_ai_llm"].value_counts(dropna=False))
        display(df_segment_labels.head(10))

    # join transcript-level labels back to transcript metadata
    cols = ["video_id", "contains_ai_llm", "quote", "quotes", "quote_count", "segment_count", "segments_with_ai"]
    df_with_labels = df.merge(
        df_labels[[c for c in cols if c in df_labels.columns]],
        on="video_id",
        how="left",
    )
    df_with_labels.head(10)

contains_ai_llm
True    6
Name: count, dtype: int64


In [9]:
# Compare keyword detection vs Ollama labels at transcript level
if "contains_keywords_ai" not in df.columns:
    if "keywords_ai" not in globals():
        keywords_ai = [
            "artificial intelligence",
            "AI",
            "AI-assisted",
            "large language models",
            "GenAI",
            "generative AI",
            "generative artificial intelligence",
        ]
    df["contains_keywords_ai"] = df["text"].str.contains(
        "|".join(keywords_ai), regex=True, case=False
    )

if "df_labels" not in globals() or df_labels.empty:
    print("Run Cell 6 first to build df_labels.")
else:
    compare_cols = ["video_id", "contains_ai_llm"]
    df_compare = df.merge(
        df_labels[compare_cols].drop_duplicates(subset=["video_id"]),
        on="video_id",
        how="left",
    )

    pd.crosstab(
        df_compare["contains_keywords_ai"],
        df_compare["contains_ai_llm"],
        rownames=["Keyword hit"],
        colnames=["Ollama label"],
        margins=True,
        dropna=False,
    )



Run Cell 6 first to build df_labels.


In [4]:
fn_to_annotate = "output/ai_to_annotate.csv"

if "df_segments" not in globals() or df_segments.empty:
    raise ValueError("Run Cell 3 first to build df_segments.")

if "contains_keywords_ai" not in df_segments.columns:
    raise ValueError("Run Cell 4 first so keyword flags exist on segments.")

# keyword-only segment candidates
df_to_annotate = df_segments[df_segments["contains_keywords_ai"].fillna(False)].copy()

# include segment text for manual review
df_to_annotate = df_to_annotate.rename(columns={"segment_text": "text"})
df_to_annotate["id"] = df_to_annotate["row_id"]

cols = [
    "id",
    "video_id",
    "segment_id",
    "date",
    "title",
    "start_hms",
    "end_hms",
    "contains_keywords_ai",
    "keyword_hit_count",
    "text",
]

df_to_annotate[cols].to_csv(fn_to_annotate, index=False)
print(f"Saved {len(df_to_annotate)} keyword-hit segments to {fn_to_annotate}")
display(df_to_annotate[cols].head(10))

Saved 751 keyword-hit segments to output/ai_to_annotate.csv


,id,video_id,segment_id,date,title,start_hms,end_hms,contains_keywords_ai,keyword_hit_count,text
1,RM0aq5ynUiQ__seg_0002,RM0aq5ynUiQ,seg_0002,2026-01-14,"""Building an AI-Ready America""",00:04:30,00:05:00,True,3,Artificial intelligence is uh scaling rapidly....
2,RM0aq5ynUiQ__seg_0003,RM0aq5ynUiQ,seg_0003,2026-01-14,"""Building an AI-Ready America""",00:05:00,00:05:30,True,3,"described as a sudden disruption, but many of ..."
3,RM0aq5ynUiQ__seg_0004,RM0aq5ynUiQ,seg_0004,2026-01-14,"""Building an AI-Ready America""",00:05:30,00:06:00,True,2,"personalize learning, and free up more time to..."
4,RM0aq5ynUiQ__seg_0005,RM0aq5ynUiQ,seg_0005,2026-01-14,"""Building an AI-Ready America""",00:06:00,00:06:30,True,2,which could mean a permanent increase in econo...
7,RM0aq5ynUiQ__seg_0008,RM0aq5ynUiQ,seg_0008,2026-01-14,"""Building an AI-Ready America""",00:07:30,00:08:00,True,2,"AI-related development, our policies must adap..."
8,RM0aq5ynUiQ__seg_0009,RM0aq5ynUiQ,seg_0009,2026-01-14,"""Building an AI-Ready America""",00:08:00,00:08:30,True,3,and prioritizing safety and privacy. US leader...
10,RM0aq5ynUiQ__seg_0011,RM0aq5ynUiQ,seg_0011,2026-01-14,"""Building an AI-Ready America""",00:09:00,00:09:30,True,2,>> From chatbots to programming to grocery lis...
11,RM0aq5ynUiQ__seg_0012,RM0aq5ynUiQ,seg_0012,2026-01-14,"""Building an AI-Ready America""",00:09:30,00:10:00,True,2,"our schools, workplaces, and healthcare settin..."
12,RM0aq5ynUiQ__seg_0013,RM0aq5ynUiQ,seg_0013,2026-01-14,"""Building an AI-Ready America""",00:10:00,00:10:30,True,3,"students, streamline assignments, and provide ..."
13,RM0aq5ynUiQ__seg_0014,RM0aq5ynUiQ,seg_0014,2026-01-14,"""Building an AI-Ready America""",00:10:30,00:11:00,True,4,have enough data right now to understand the f...


Ok this is the code Leon made to have his little yes/no pop up window, I haven't been able to run it because my videos have like 100,000 mentions of AI (naturally) so its not really helpful for me at this point. But! you should try it 

In [ ]:
import csv
import os
import random
import sys

fn_items = "output/ai_to_annotate.csv"
fn_labels = "output/ai_human_labels.csv"

SEED = 303
fieldnames = ["id", "video_id", "segment_id", "contains_ai_human", "interesting", "reviewer", "note"]
answers = {"t": "True", "f": "False", "s": ""}

reviewer = input("your name > ").strip()
video_filter = input("video_id to review (blank = all) > ").strip()

# one row per segment, shuffled reproducibly
with open(fn_items, encoding="utf-8", newline="") as f:
    items = list(csv.DictReader(f))
random.Random(SEED).shuffle(items)

if video_filter:
    items = [item for item in items if item.get("video_id") == video_filter]
    if not items:
        print(f"No rows found for video_id={video_filter}")

# skip segments this reviewer already labeled
done = set()
if os.path.exists(fn_labels):
    with open(fn_labels, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if row.get("reviewer") == reviewer:
                done.add(row.get("id"))

queue = [item for item in items if item.get("id") not in done]
print(f"{len(items) - len(queue)} of {len(items)} already labeled by {reviewer}")

os.makedirs("output", exist_ok=True)
is_new = not os.path.exists(fn_labels)
f = open(fn_labels, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(f, fieldnames=fieldnames)
if is_new:
    writer.writeheader()
    f.flush()

history = []  # (position in queue, file size before writing) for undo
i = 0
while i < len(queue):
    item = queue[i]
    row_id = item.get("id", "")
    video_id = item.get("video_id", "")
    segment_id = item.get("segment_id", "")
    url = f"https://www.youtube.com/watch?v={video_id}"

    print("\n" + "-" * 70)
    print(f"{len(items) - len(queue) + i + 1} of {len(items)}  |  {url}")
    print(f"SEGMENT: {segment_id}  ({item.get('start_hms', '?')} - {item.get('end_hms', '?')})")
    print(f"TITLE: {item.get('title', '').strip()}")

    text = (item.get("text") or "").strip()
    print(text[:1800] + ("\n...[truncated]" if len(text) > 1800 else ""))
    print("-" * 70)

    reply = input("mentions AI/LLMs? [t/f/s, add i = interesting, b = undo, q = quit] > ").strip()
    key, _, note = reply.partition(" ")
    key = key.lower()

    interesting = "i" in key and key != "i"
    key = key.replace("i", "") if interesting else key

    if key == "q":
        break
    if key == "b":
        if history:
            i, size = history.pop()
            f.truncate(size)
            f.seek(0, os.SEEK_END)
        else:
            print("nothing to undo")
        continue
    if key not in answers:
        print("answer t, f, s (add i for interesting), b or q")
        continue

    history.append((i, f.tell()))
    writer.writerow({
        "id": row_id,
        "video_id": video_id,
        "segment_id": segment_id,
        "contains_ai_human": answers[key],
        "interesting": "True" if interesting else "",
        "reviewer": reviewer,
        "note": note,
    })
    f.flush()
    i += 1

f.close()
print(f"labels saved to {fn_labels}")

This code is for comparing annotations by different people, guess we need the above code to run first? 

In [ ]:
# Load human AI labels (segment-level) and compare across reviewers
import csv
import sys

# allow very long text fields in CSV
max_int = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_int)
        break
    except OverflowError:
        max_int = int(max_int / 10)

fn_human_labels = "output/ai_human_labels.csv"

if not os.path.exists(fn_human_labels):
    print(f"No labels file found at {fn_human_labels}. Run Cell 11 first.")
else:
    df_annotations = pd.read_csv(fn_human_labels, dtype=str, keep_default_na=False)

    if df_annotations.empty:
        print("Labels file is empty.")
    else:
        # normalize values
        df_annotations["contains_ai_human"] = df_annotations["contains_ai_human"].replace("", "skip")

        # one row per segment, one column per reviewer
        index_cols = [c for c in ["id", "video_id", "segment_id"] if c in df_annotations.columns]
        df_wide = df_annotations.pivot_table(
            index=index_cols,
            columns="reviewer",
            values="contains_ai_human",
            aggfunc="first",
            fill_value="skip",
        )

        col_reviewers = df_wide.columns.tolist()
        print(f"Loaded {len(df_annotations)} human labels from {len(col_reviewers)} reviewer(s).")
        df_wide.head()

In [ ]:
has_skip = (df_wide[col_reviewers] == "skip").any(axis=1)
has_same_answer = df_wide[col_reviewers].nunique(axis=1) == 1
df_wide["is_unanimous"] = has_same_answer & ~has_skip

answered = df_wide[col_reviewers].isin(["True", "False"]).sum()
share_true = (df_wide[col_reviewers] == "True").sum() / answered.replace(0, pd.NA)

print(f"Reviewers agreed on {df_wide['is_unanimous'].mean():.0%} of segments")
if share_true.notna().any():
    print(
        f"Between {share_true.min(skipna=True):.0%} and {share_true.max(skipna=True):.0%} "
        "of segments mention AI/LLMs, depending on the reviewer"
    )

In [ ]:
df_text = pd.read_csv(fn_to_annotate, dtype=str).set_index("id")

df_review = (
    df_wide.loc[~df_wide["is_unanimous"], col_reviewers]
    .join(df_text[["video_id", "segment_id", "start_hms", "end_hms", "text"]], how="left")
)

df_review.head(3)